In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
import os
import pulp

In [10]:
# GAMMA = 5.31  # xj
# ALPHA = 0.60  # yj
# BETA = 0.10   # zj
# GAMMA = 50  # xj
# ALPHA = 15  # yj
# BETA = 5   # zj
GAMMA = 0.18  # xj
ALPHA = 0.10  # yj
BETA = 0.07  # zj
LAMBDA = 0.001 
R_LIMIT = 2000 
BUDGET_ACTIONS = 15 # suppose we can only implement 15 actions in total

COST_NEW = 5   
COST_UPGRADE = 3.0
COST_RENO = 2.0
TOTAL_BUDGET = 100.0
SEARCH_RADIUS = 1000

In [11]:
gdf_res = gpd.read_file(r'./dataset_structured/05_processed/standardized_layers/elderly_demand_3857.shp')
gdf_parks = gpd.read_file(r'./dataset_structured/05_processed/standardized_layers/parks_with_quality_3857.shp')
gdf_candidates = gpd.read_file(r'./dataset_structured/05_processed/standardized_layers/candidate_new_park_spots.shp')

res_coords = np.array(list(gdf_res.geometry.apply(lambda p: (p.x, p.y))))
elderly_dem = gdf_res['elderly_de'].values

In [12]:
def calculate_action_gain(action_coord, q_gain, current_res_coords, dem_values):
    """计算单个动作对全市 Di * Ai 的边际贡献"""
    dists = np.linalg.norm(current_res_coords - action_coord, axis=1)
    mask = dists <= R_LIMIT
    if not np.any(mask): return 0
    
    # delta_Ai = q_gain * exp(-lambda * dist)
    delta_ai = q_gain * np.exp(-LAMBDA * dists[mask])
    # total gain = sum( Di * delta_Ai )
    return np.sum(dem_values[mask] * delta_ai)

In [13]:
action_pool = []

# 1. New Park (xj)
for i, row in gdf_candidates.iterrows():
    coord = np.array([row.geometry.x, row.geometry.y])
    gain = calculate_action_gain(coord, GAMMA, res_coords, elderly_dem)
    
    action_pool.append({
        'uid': f'New_{i}', 
        'location_id': f'Spot_{i}', 
        'type': 'xj_NewPark', 
        'coord': coord, 
        'q_gain': GAMMA, 
        'total_gain': gain,
        'cost': COST_NEW,
        'geometry': row.geometry
    })

# 2. Upgrade (yj) & Renovation (zj)
for i, row in gdf_parks.iterrows():
    coord = np.array([row.geometry.centroid.x, row.geometry.centroid.y])
    park_name = row.get('name', f'Park_{i}')

    gain_up = calculate_action_gain(coord, ALPHA, res_coords, elderly_dem)
    gain_reno = calculate_action_gain(coord, BETA, res_coords, elderly_dem)

    action_pool.append({
        'uid': f'Up_{i}', 
        'location_id': park_name,
        'type': 'yj_Upgrade', 
        'coord': coord, 
        'q_gain': ALPHA, 
        'total_gain': gain_up, 
        'cost': COST_UPGRADE, 
        'geometry': row.geometry.centroid
    })
    
    action_pool.append({
        'uid': f'Reno_{i}', 
        'location_id': park_name, 
        'type': 'zj_Renovation', 
        'coord': coord, 
        'q_gain': BETA, 
        'total_gain': gain_reno, 
        'cost': COST_RENO, 
        'geometry': row.geometry.centroid
    })

In [14]:
prob = pulp.LpProblem("Park_Optimization", pulp.LpMaximize)

# 3. create decision variables
x_vars = pulp.LpVariable.dicts("X", [a['uid'] for a in action_pool], cat='Binary')

# 4. max objective: maximize total gain
prob += pulp.lpSum([a['total_gain'] * x_vars[a['uid']] for a in action_pool])

# 5. constraint A: total cost within budget
prob += pulp.lpSum([a['cost'] * x_vars[a['uid']] for a in action_pool]) <= TOTAL_BUDGET

# 6. constraint B: mutual exclusion constraints (important!)
# e.g., the same park cannot be both upgraded (Up) and renovated (Reno)
park_ids = set([a['geometry'] for a in action_pool if 'geometry' in a])
for pid in park_ids:
    related_acts = [a['uid'] for a in action_pool if a.get('geometry') == pid]
    if len(related_acts) > 1:
        prob += pulp.lpSum([x_vars[uid] for uid in related_acts]) <= 1

# 7. constraint C: spatial exclusivity for new parks (optional, can be relaxed)
new_park_pool = [a for a in action_pool if a['type'] == 'xj_NewPark']
if len(new_park_pool) > 0:
    new_coords = np.array([a['coord'] for a in new_park_pool])
    new_uids = [a['uid'] for a in new_park_pool]
    
    tree = cKDTree(new_coords)
    pairs = tree.query_pairs(r=SEARCH_RADIUS)
    
    for i, j in pairs:
        prob += x_vars[new_uids[i]] + x_vars[new_uids[j]] <= 1

print("Solving ILP...")
prob.solve(pulp.PULP_CBC_CMD(msg=1)) 

print(f"Status: {pulp.LpStatus[prob.status]}")

Solving ILP...
Status: Optimal


In [15]:
selected_results = []
for a in action_pool:
    if pulp.value(x_vars[a['uid']]) == 1:
        selected_results.append(a)


In [16]:
for res in selected_results:
    print(f"Selected Action: {res['type']} at {res['location_id']} with gain {res['total_gain']:.2f} and cost {res['cost']:.2f}")

Selected Action: xj_NewPark at Spot_116 with gain 463.06 and cost 5.00
Selected Action: xj_NewPark at Spot_147 with gain 401.89 and cost 5.00
Selected Action: xj_NewPark at Spot_155 with gain 397.18 and cost 5.00
Selected Action: xj_NewPark at Spot_166 with gain 516.93 and cost 5.00
Selected Action: xj_NewPark at Spot_189 with gain 398.92 and cost 5.00
Selected Action: xj_NewPark at Spot_218 with gain 419.84 and cost 5.00
Selected Action: xj_NewPark at Spot_220 with gain 503.82 and cost 5.00
Selected Action: xj_NewPark at Spot_224 with gain 359.85 and cost 5.00
Selected Action: xj_NewPark at Spot_230 with gain 388.83 and cost 5.00
Selected Action: xj_NewPark at Spot_232 with gain 348.66 and cost 5.00
Selected Action: zj_Renovation at 中山公园 with gain 144.39 and cost 2.00
Selected Action: yj_Upgrade at 四川北路公园 with gain 277.06 and cost 3.00
Selected Action: zj_Renovation at 番禺绿地 with gain 155.27 and cost 2.00
Selected Action: zj_Renovation at 0 with gain 142.72 and cost 2.00
Selected Actio

In [17]:
gdf_ilp = gpd.GeoDataFrame(selected_results, crs="EPSG:3857")
gdf_ilp.to_file(r'./dataset_structured/05_processed/standardized_layers/optimized_decisions_v2.shp')
print(f"Done! Selected {len(selected_results)} optimal actions.")

Done! Selected 33 optimal actions.


C:\Users\laura\AppData\Local\Temp\ipykernel_36280\701615106.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_ilp.to_file(r'./dataset_structured/05_processed/standardized_layers/optimized_decisions_v2.shp')
c:\Users\laura\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'location_id' to 'location_i'
  ogr_write(
